# DSPy + LangChain + llama.cpp (OpenAI-Compatible)
Este notebook demonstra como usar um modelo rodando via **llama.cpp** como servidor OpenAI-compatible com **DSPy + GEPA**.

In [ ]:
import os

# Configure llama.cpp server
os.environ['OPENAI_API_KEY'] = 'token-abc123'
os.environ['OPENAI_BASE_URL'] = 'http://localhost:8080/v1'

print('Environment configured for llama.cpp OpenAI-compatible server.')

## Importações principais

In [ ]:
from typing import Any, Callable, List, Type, Union
import dspy
from dspy import Signature
from langchain_openai import ChatOpenAI
from langchain_core.runnables import Runnable
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage

## Implementação completa do LangChainAgentModuleFactory

In [ ]:
class LangChainAgentModuleFactory(dspy.Module):
    def __init__(self, signature, agent_factory, system_prompt_field="system_prompt"):
        super().__init__()
        self.signature = signature
        self.agent_factory = agent_factory
        self.system_prompt_field = system_prompt_field
        self.predictor = dspy.Predict(signature)
        self.initial_system_prompt = signature.__doc__.strip() if getattr(signature, "__doc__", None) else ""

    def _extract_system_prompt(self):
        sig = getattr(self.predictor, "extended_signature", self.predictor.signature)
        if hasattr(sig, "instructions") and sig.instructions:
            return sig.instructions.strip()
        if getattr(sig, "__doc__", None):
            return sig.__doc__.strip()
        return self.initial_system_prompt

    def _prepare_langchain_messages(self, system_prompt, **kwargs):
        messages = [SystemMessage(content=system_prompt)]
        parts = [f"### {k}: {v}\n" for k, v in kwargs.items() if v]
        if parts:
            messages.append(HumanMessage(content="\n".join(parts)))
        return messages

    def _extract_response(self, result):
        if isinstance(result, str):
            return result
        if isinstance(result, dict):
            for key in ["output", "response", "answer", "content", "text"]:
                if key in result:
                    val = result[key]
                    if hasattr(val, "content"):
                        return val.content
                    if isinstance(val, dict):
                        return val.get("content", str(val))
                    return str(val)
            if "messages" in result:
                last = result["messages"][-1]
                return getattr(last, "content", str(last))
        if hasattr(result, "content"):
            return result.content
        if isinstance(result, list) and result:
            last = result[-1]
            return getattr(last, "content", str(last))
        return str(result)

    def _get_output_fields(self):
        if isinstance(self.signature, str):
            return [self.signature.split("->")[-1].strip()]
        fields = []
        if hasattr(self.signature, "output_fields"):
            for name, info in self.signature.output_fields.items():
                if getattr(info, "json_schema_extra", {}).get("_dspy_field_type") == "output":
                    fields.append(name)
        return fields or ["response"]

    def forward(self, **kwargs):
        system_prompt = self._extract_system_prompt()
        agent = self.agent_factory()
        messages = self._prepare_langchain_messages(system_prompt, **kwargs)

        try:
            result = agent.invoke({"messages": messages})
            if not isinstance(result, dict):
                raise TypeError(f"Agent must return dict, got {type(result)}")

            response = self._extract_response(result)
            output_field = self._get_output_fields()[0]

            original_forward = self.predictor.forward
            def mock_forward(**pred_kwargs):
                return dspy.Prediction({output_field: response})
                #return dspy.Prediction(response=response)

            try:
                self.predictor.forward = mock_forward
                return self.predictor(**kwargs)
            finally:
                self.predictor.forward = original_forward

        except Exception as e:
            raise RuntimeError(f"Agent failed: {e}\nInputs: {kwargs}")

## Criar assinatura DSPy

In [ ]:
import dspy
from dspy import Signature

class ClassifyIssue(Signature):
    """Classifique a seguinte frase (issue) entre as seguintes categorias: billing, account and other. Retorne somente a categoria"""
    
    issue = dspy.InputField()
    response = dspy.OutputField()


## Criar agente LangChain usando ChatOpenAI apontando para llama.cpp

In [ ]:
class LlamaOpenAIProxy(Runnable):
    def __init__(self, model="gpt-4o-mini", max_tokens=256):
        self.llm = ChatOpenAI(
            model=model,
            temperature=0.2,
            max_tokens=max_tokens
        )

    def invoke(self, inputs):
        messages = inputs["messages"]
        # Como estou usando modelo open source, o system message fica bem ruim. Vou combinar ambas mensagens em apenas uma só
        if isinstance(messages[0], SystemMessage):
            messages = messages[0].content + messages[1].content
            
        output = self.llm.invoke(messages)
        return {
            "response": output.content,
            "messages": [AIMessage(content=output.content)]
        }

def llama_agent_factory():
    return LlamaOpenAIProxy()

## Instanciar módulo DSPy

In [ ]:
module = LangChainAgentModuleFactory(
    signature=ClassifyIssue,
    agent_factory=llama_agent_factory
)
module

In [ ]:
module(issue = "Eu esqueci minha senha")

## Criar dataset de treino e validação

In [ ]:
trainset = [
    dspy.Example(issue="I want a refund for my order", response="billing").with_inputs("issue"),
    dspy.Example(issue="I forgot my password", response="account").with_inputs("issue"),
    dspy.Example(issue="The app crashes sometimes", response="other").with_inputs("issue"),
]


valset = trainset

## Métrica GEPA

In [ ]:
def gepa_metric(*args):
    if len(args) == 2:
        gold, pred = args
    elif len(args) == 3:
        gold, pred, trace = args
    elif len(args) == 5:
        gold, pred, trace, pred_name, pred_trace = args
    else:
        raise ValueError(f"Unexpected metric signature: {len(args)} args")

    pred_label = getattr(pred, "response", None)
    if pred_label is None:
        return 0.0

    return 1.0 if gold.response == pred_label else 0.0

In [ ]:
reflection_lm = dspy.LM(
    model="gpt-4o-mini",   # nome não importa, llama.cpp ignora
    temperature=0.7,
    max_tokens=2048
)

In [ ]:
reflection_lm("Diga oi")

## Rodar GEPA Optimization

In [ ]:
optimizer = dspy.GEPA(
    metric=gepa_metric,
    auto="light",
    reflection_lm=reflection_lm
)
optimized_module = optimizer.compile(module, trainset=trainset, valset=valset)

optimized_module

## Testar módulo otimizado

In [ ]:
optimized_module(issue="I need a refund for my subscription")